# Markdown file to only look at summary.json files

In [1]:
DEFAULT_SUMMARY_PATH = "//dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/{input_dataset}/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/{fold}/{validation}/summary.json"

import pandas as pd

path_902 = DEFAULT_SUMMARY_PATH.format(input_dataset="Dataset902_AutoPet", fold="{fold}", validation="{validation}")
path_903 = DEFAULT_SUMMARY_PATH.format(input_dataset="Dataset903_AutoPet", fold="{fold}", validation="{validation}")
path_901 = DEFAULT_SUMMARY_PATH.format(input_dataset="Dataset901_AutoPet", fold="{fold}", validation="{validation}")

def read_summary_file(path, fold, validation): 
    import json
    with open(path.format(fold=fold, validation=validation), "r") as f:
        return json.load(f)

f902_0 = read_summary_file(path_902, fold="fold_0", validation="validation")
f902_1 = read_summary_file(path_902, fold="fold_1", validation="validation_279")
f902_2 = read_summary_file(path_902, fold="fold_2_first", validation="validation_279")

f903_0 = read_summary_file(path_903, fold="fold_0", validation="validation_279")
f903_1 = read_summary_file(path_903, fold="fold_1", validation="validation")
f903_2 = read_summary_file(path_903, fold="fold_2", validation="validation_279")

f901_6 = read_summary_file(path_901, fold="older_iterations/fold_6", validation="validation_with_mirror")
f901_7 = read_summary_file(path_901, fold="older_iterations/fold_7", validation="validation_with_mirror")
f901_8 = read_summary_file(path_901, fold="older_iterations/fold_8", validation="validation")
# f901_9 = read_summary_file(path_901, fold="fold_9")
# f901_10 = read_summary_file(path_901, fold="fold_10")

# get variable name of f0 
summary_files = [f902_0, f902_1, f902_2, f903_0, f903_1, f903_2, f901_6, f901_7, f901_8]
names = ["f902_0", "f902_1", "f902_2", "f903_0", "f903_1", "f903_2", "f901_6", "f901_7", "f901_8"]
names_dict= {
    "f902_0": "active learning 1% (no tta)",
    "f902_1": "active learning 1% with 15% pseudo labels (tta)",
    "f902_2": "active learning 1% with 30% pseudo labels (tta)",
    "f903_0": "active_learning 5% (tta)",
    "f903_1": "active_learning 5% with 15% pseudo labels (tta)",
    "f903_2": "active_learning 5% with 30% pseudo labels (tta)",
    "f901_6": "All of the pseudo labels (tta)",
    "f901_7": "15% pseudo labels (tta)",
    "f901_8": "30% pseudo labels (no tta)"
}

names = [names_dict[name] for name in names]
dfs = []

for i, summary_file in enumerate(summary_files):
    df = pd.DataFrame(summary_file['foreground_mean'], index=[0]).round(3)
    df["fold"] = names[i]
    dfs.append(df)

summary_df = pd.concat(dfs, ignore_index=True)
# make fold the first column 
cols = summary_df.columns.tolist()
cols = cols[-1:] + cols[:-1]
summary_df = summary_df[cols]
summary_df.sort_values('Dice')

FileNotFoundError: [Errno 2] No such file or directory: '//dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset901_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/older_iterations/fold_6/validation_with_mirror/summary.json'

# Test Data

In [14]:
path_902 = DEFAULT_SUMMARY_PATH.format(input_dataset="Dataset901_AutoPet", fold="{fold}", validation="{validation}")

dats = []
names = ["fold_0", "fold_1", "fold_2"]
for fold in ["fold_0", "fold_1", "fold_2"]:
    temp = read_summary_file(path_902, fold=fold, validation="test_predictions")
    dats.append(temp)

names_dict = {
    "fold_0": "11% active learning (tta)",
    "fold_1": "11% active learning with 15% pseudo labels (tta)",
    "fold_2": "11% active learning with 30% pseudo labels (tta)"
}

# make dataframe for fold_1, fold_5, fold_20, fold_30
dfs_902 = []
for i, fold in enumerate(dats):
    df = pd.DataFrame(fold['foreground_mean'], index=[0]).round(3)
    df["fold"] = names_dict[names[i]]
    dfs_902.append(df)
summary_df_902 = pd.concat(dfs_902, ignore_index=True)
# make fold the first column
cols = summary_df_902.columns.tolist()
cols = cols[-1:] + cols[:-1]
summary_df_902 = summary_df_902[cols]
summary_df_902.sort_values('Dice')


FileNotFoundError: [Errno 2] No such file or directory: '//dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset901_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_0/test_predictions/summary.json'

In [17]:
import os


def load_dataset_folds(dataset_num, folds=[f'fold_{i}' for i in range(0, 9)], validation="validation_279"):
    path = DEFAULT_SUMMARY_PATH.format(
        input_dataset=f"Dataset{dataset_num}_AutoPet",
        fold="{fold}",
        validation="{validation}"
    )
    
    summary_dfs = []
    fdg_dfs = []
    psma_dfs = []
    
    for i in folds:
        filepath = path.format(fold=i, validation=validation)
        try:
            fold_data = read_summary_file(path, fold=i, validation=validation)
            
            # --- Per-case metrics, split by tracer ---
            fdg_rows = []
            psma_rows = []
            
            for case in fold_data['metric_per_case']:
                metrics = case['metrics']['1']
                pred_file = case['prediction_file']
                case_name = os.path.basename(pred_file).replace('.nii.gz', '')
                
                row = {
                    "Dice": metrics['Dice'],
                    "FP":   metrics['FP'],
                    "FN":   metrics['FN'],
                }
                
                if case_name.lower().startswith('fdg'):
                    fdg_rows.append(row)
                elif case_name.lower().startswith('psma'):
                    psma_rows.append(row)
            
            def make_summary_row(rows, fold_label):
                if not rows:
                    return None
                df = pd.DataFrame(rows)
                return pd.DataFrame({
                    "fold":          [fold_label],
                    "Dice":          [round(df['Dice'].mean(), 4)],
                    "FP (voxels)":   [int(df['FP'].sum())],
                    "FN (voxels)":   [int(df['FN'].sum())],
                    "n_cases":       [len(rows)],
                })
            
            # Overall summary from foreground_mean
            df_summary = pd.DataFrame({
                "fold":        [str(i)],
                "Dice":        [round(fold_data['foreground_mean']['Dice'], 4)],
                "FP (voxels)": [sum(c['metrics']['1']['FP'] for c in fold_data['metric_per_case'])],
                "FN (voxels)": [sum(c['metrics']['1']['FN'] for c in fold_data['metric_per_case'])],
                "n_cases":     [len(fold_data['metric_per_case'])],
            })
            summary_dfs.append(df_summary)
            
            row_fdg  = make_summary_row(fdg_rows,  str(i))
            row_psma = make_summary_row(psma_rows, str(i))
            if row_fdg  is not None: fdg_dfs.append(row_fdg)
            if row_psma is not None: psma_dfs.append(row_psma)
            
        except Exception as e:
            print(f"Could not read fold {i} for dataset {dataset_num}")
            print(f"Expected path: {filepath}")
            print(f"Error: {e}")
    
    summary_df = pd.concat(summary_dfs, ignore_index=True).sort_values('fold')
    fdg_df     = pd.concat(fdg_dfs,     ignore_index=True).sort_values('fold') if fdg_dfs  else None
    psma_df    = pd.concat(psma_dfs,    ignore_index=True).sort_values('fold') if psma_dfs else None
    
    return summary_df, fdg_df, psma_df


def display_table(df, title, cmap_dice='RdYlGn', cmap_err='RdYlGn_r'):
    print(f"\n=== {title} ===")
    display(df.style
        .background_gradient(subset=['Dice'],        cmap=cmap_dice)
        .background_gradient(subset=['FP (voxels)', 'FN (voxels)'], cmap=cmap_err)
        .format({
            'Dice':        '{:.4f}',
            'FP (voxels)': '{:,}',
            'FN (voxels)': '{:,}',
            'n_cases':     '{:,}',
        })
    )


# --- Run ---
summary_df, fdg_df, psma_df = load_dataset_folds(
    901, folds=[f"older_iterations/fold_{i}" for i in range(0, 11)], validation="validation_279"
)

display_table(summary_df, "Overall Summary")
if fdg_df  is not None: display_table(fdg_df,  "FDG Only")
if psma_df is not None: display_table(psma_df, "PSMA Only")

Could not read fold older_iterations/fold_0 for dataset 901
Expected path: //dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset901_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/older_iterations/fold_0/validation_279/summary.json
Error: [Errno 2] No such file or directory: '//dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset901_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/older_iterations/fold_0/validation_279/summary.json'
Could not read fold older_iterations/fold_1 for dataset 901
Expected path: //dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset901_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/older_iterations/fold_1/validation_279/summary.json
Error: [Errno 2] No such file or directory: '//dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset901_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/old

,fold,Dice,FP (voxels),FN (voxels),n_cases
0,older_iterations/fold_6,0.6341,"963,639","388,197",247
1,older_iterations/fold_7,0.5758,"722,733","748,637",247
2,older_iterations/fold_8,0.6153,"635,999","621,663",247



=== FDG Only ===


,fold,Dice,FP (voxels),FN (voxels),n_cases
0,older_iterations/fold_6,0.7188,"683,725","248,005",151
1,older_iterations/fold_7,0.6633,"351,050","629,978",151
2,older_iterations/fold_8,0.7236,"325,799","454,837",151



=== PSMA Only ===


,fold,Dice,FP (voxels),FN (voxels),n_cases
0,older_iterations/fold_6,0.5565,"279,914","140,192",96
1,older_iterations/fold_7,0.4956,"371,683","118,659",96
2,older_iterations/fold_8,0.5161,"310,200","166,826",96
